# BraMex · Team notebook
**MITxT 2.980x Sports Analytics · Soccer project**
Leonardo De Andrade (PM & integration lead) · Monica D. Morales-Hernandez (modeling lead) · Pablo Saraiva (evaluation & validation lead)

**Project question:** what does physical exertion buy in a football match? We use 25 Hz tracking and event data from six DFL matches, and a final model is evaluated on a match nobody has seen.

This notebook is the single place where the team's work comes together. Each part says **what it must deliver**, **who owns it** and **what the next part expects from it**. The code is written by each owner inside their own part.

## How to use this notebook

**Structure**

| Part | Owner | Delivers |
|---|---|---|
| 1 · Setup & configuration | Leonardo | Environment, paths, agreed parameters, holdout guard |
| 2 · Data loading | Leonardo | Raw data for every match, cached |
| 3 · Shared tables | Leonardo · Monica | The four standard tables every workstream uses |
| 4 · Data checks | Pablo | Checks that must pass before any analysis |
| 5 · Workstream A - players & physical load | Leonardo | Findings + features |
| 6 · Workstream B - teams: possession, progression, defense | Monica | Findings + features |
| 7 · Workstream C - chances & possession changes | Pablo | Findings + features |
| 8 · Integration | Leonardo | One feature table from all workstreams |
| 9 · Prediction | Monica · Pablo | Validated, frozen model |
| 10 · Holdout inference | Pablo | Predictions on the unseen match |
| 11 · Results log | All | Every claim with its evidence |
| 12 · Conclusions & limitations | All | What we can and cannot say |

**Team rules**
1. **Work only in your own part.** Changes to Parts 1-4 are agreed with the PM first, because every result depends on them.
2. **Parameters come from the decision log.** No threshold, window or model setting is typed inside a workstream; it is read from the configuration in Part 1.
3. **Same tables for everyone.** Workstreams start from the tables of Part 3, never from their own copy of the raw files.
4. **The holdout match stays closed** until the models are frozen (Part 9.8).
5. **Every claim carries an evidence label** and goes into the results log (Part 11).
6. **Clear all outputs before committing.** Results stay inside the team until we have reviewed them.

**Evidence labels**

| Label | Meaning |
|---|---|
| Descriptive | What the data shows, no relationship claimed |
| Association | Two quantities move together within the stated unit and controls |
| Robust association | The association holds across matches, alternative definitions and placebo tests |
| Predictive (out-of-match) | Beats a baseline on matches the model did not see |
| Causal (not claimed) | We do not claim causality; use this label only to state that explicitly |

## 1 · Setup & configuration
**Owner:** Leonardo · **Delivers:** a working environment and one configuration object used by every part.

### 1.1 Environment
Everyone must be able to run the notebook from top to bottom on their own machine or on Google Colab.
- Python 3.11+ with `floodlight 1.2.0`, `numpy`, `pandas`, `scipy`, `scikit-learn`, `statsmodels`, `matplotlib`.
- Imports, display options (no scientific notation in outputs) and a fixed random seed.

In [ ]:
# Imports, display options and random seed


### 1.2 Paths
The data folder is read from the environment variable `BRAMEX_DATA_DIR`, so nobody edits paths in the notebook. Cache and model folders are created if they do not exist.

In [ ]:
# Data, cache and model paths


### 1.3 Parameters from the decision log
One configuration object with every parameter that changes results. Each value points to its decision and its status (Proposed / Agreed).

| Decision | Parameter(s) | Owner |
|---|---|---|
| D1 Smoothing of positions | Savitzky–Golay window and order | Leonardo |
| D2 Load metric | HSR, sprint, metabolic power, ∫a² dt | Leonardo |
| D3 Speed and acceleration thresholds | HSR speed, sprint speed, acceleration | Leonardo |
| D4 Pitch-control model | Voronoi and/or time-to-arrival; grid, ball weighting, model parameters | Monica |
| D5 Time windows | Sampling rate and aggregation windows | All |
| D6 Possession definition | DFL flag or own rule | Monica |
| D7 Prediction target and metric | Must match the course's holdout evaluation | All |
| D8 Validation scheme | Leave-one-match-out or alternative | Pablo |

In [ ]:
# Configuration object with every decision-log parameter and its status


### 1.4 Holdout guard
A list of match IDs that cannot be loaded before the freeze. Any function that loads data must refuse these IDs.

In [ ]:
# Holdout list and guard


## 2 · Data loading
**Owner:** Leonardo · **Delivers:** the raw tracking, possession, ball status, events and team sheets for each match, cached on disk.

Each match has three DFL files, identified by the match ID at the end of the name (`DFL-MAT-XXXXXX`):
- **Positions** (`positions_raw_observed`) - 25 Hz x/y for every player and the ball, plus possession and ball-status flags.
- **Match information** (`matchinformation`) - team sheets: players, shirt numbers, positions, team IDs.
- **Events** (`events_raw`) - passes, shots, tackles, fouls, substitutions, with time and players.

### 2.1 Discover matches
List every match ID with its three files. Report incomplete matches instead of failing silently.

In [ ]:
# Find matches and their three files


### 2.2 Read and cache one match
Parse with `floodlight` once and save to the cache; later runs read the cache. Must respect the holdout guard.

In [ ]:
# Read one match with floodlight and cache it


### 2.3 Load all analysis matches
Loop over every non-holdout match and report what was loaded (halves, frames, players per team).

In [ ]:
# Load all non-holdout matches


## 3 · Shared tables
**Owners:** Leonardo (load) · Monica (pitch control and possession) · **Delivers:** four standard tables, identical for every workstream and for the holdout match.

| Table | One row per | Must contain |
|---|---|---|
| `samples` | player × analysis sample (ball in play) | match, half, time, minute, player ID, side, in possession, speed, load per sample, pitch-control measures, distance to ball, x in attacking direction |
| `team_samples` | match × analysis sample | match, half, time, minute, ball in play, possession side, ball x/y, attacking direction, team pitch-control shares |
| `events` | DFL event | match, half, time, matching sample, side, team, player, event type, outcome, location |
| `players` | player × match | player ID, name, team, side, position, position group |

Conventions: metres, origin at the centre spot; `side` is Home/Away; minutes run 0–45 in the first half and 45–90+ in the second.

### 3.1 Kinematics and physical load
Smooth positions (D1), derive speed and acceleration, then compute load per sample: distance, HSR, sprint distance, time accelerating/decelerating, metabolic power (D2, D3). Explain any cap or cleaning rule applied.

In [ ]:
# Speed, acceleration and load per sample


### 3.2 Possession, ball status and attacking direction
Keep only ball-in-play samples for analysis. Possession follows D6. Attacking direction per team and half, so every location can be expressed towards the opponent's goal.

In [ ]:
# Possession, ball in play and attacking direction


### 3.3 Pitch control
Per sample: each player's Voronoi area, ball-weighted control and, if agreed in D4, time-to-arrival control; team shares of the pitch and of the ball zone. State the model parameters and why.

In [ ]:
# Pitch control per player and per team


### 3.4 Events
Align events to tracking samples and flag the event types the workstreams need (shots, goals, passes, crosses, tackles, fouls, possession losses).

In [ ]:
# Event table aligned to samples


### 3.5 Aggregations used by all workstreams
- `player_minute`: player × minute × phase (in / out of possession).
- `team_minute`: team × minute - possession share, control, load, shots.
Any other aggregation stays inside the workstream that needs it.

In [ ]:
# player_minute and team_minute


## 4 · Data checks
**Owner:** Pablo · **Delivers:** a table of checks, all passing, before any analysis. The same checks run on the holdout match.

Minimum checks:
- Frame rate 25 Hz and pitch 105 × 68 m in every match.
- At most 11 players per team on the pitch at any sample.
- Speeds within a physical range after smoothing.
- Voronoi areas of all players add up to the pitch area.
- Possession assigned in almost every ball-in-play sample.
- Events fall inside the half and match a tracking sample.
- No holdout match in any table.

In [ ]:
# Checks table; stop if any check fails


## 5 · Workstream A - players & physical load
**Owner:** Leonardo De Andrade

**Question:** Is individual exertion linked to pitch control? Do the load metrics agree? Is fatigue visible?

Fill in before running the main analysis:

| | |
|---|---|
| Hypothesis | |
| Unit of analysis | |
| Method | |
| Controls | |
| Rejection rule | What result would make us drop the hypothesis |
| Expected figure | |

Starting points: within-player comparison of high-speed running and ball-zone control by phase; event study around high-speed efforts; agreement between load metrics; load by 15-minute block.

### 5.1 Exploration
First look at the variables: distributions, outliers, differences between matches and positions. Nothing here is a finding yet.

In [ ]:
# Exploration


### 5.2 Main analysis
The analysis that answers the question, using the method written above. Report effect sizes with confidence intervals, not only p-values.

In [ ]:
# Main analysis


### 5.3 Robustness
Does the result survive: each match separately, leave-one-match-out, alternative definitions from the decision log, a placebo or shuffled comparison, correction for multiple tests?

In [ ]:
# Robustness checks


### 5.4 Figure
One figure that a coach could read in ten seconds. Title states the finding; axes have units.

In [ ]:
# Main figure


### 5.5 Features for integration
Variables from this workstream that may help predict the target. Rules:
- Keys: `match, half, minute, side` (team level) or the same plus player ID (player level).
- Column names start with `A_`.
- Use only information available **up to and including** that minute.
- Write a one-line description of each feature.

In [ ]:
# Feature table and its registration


### 5.6 Takeaways
Two or three sentences, each with its evidence label and the number behind it. Add each one to the results log (Part 11).

## 6 · Workstream B - teams: possession, progression, defense
**Owner:** Monica D. Morales-Hernandez

**Question:** How do possession, ball progression and where a team wins the ball back relate to pitch control?

Fill in before running the main analysis:

| | |
|---|---|
| Hypothesis | |
| Unit of analysis | |
| Method | |
| Controls | |
| Rejection rule | What result would make us drop the hypothesis |
| Expected figure | |

Starting points: possession share vs pitch and ball-zone control per minute; progression of the ball towards goal per possession; location of ball recoveries; team running in and out of possession.

### 6.1 Exploration
First look at the variables: distributions, outliers, differences between matches and positions. Nothing here is a finding yet.

In [ ]:
# Exploration


### 6.2 Main analysis
The analysis that answers the question, using the method written above. Report effect sizes with confidence intervals, not only p-values.

In [ ]:
# Main analysis


### 6.3 Robustness
Does the result survive: each match separately, leave-one-match-out, alternative definitions from the decision log, a placebo or shuffled comparison, correction for multiple tests?

In [ ]:
# Robustness checks


### 6.4 Figure
One figure that a coach could read in ten seconds. Title states the finding; axes have units.

In [ ]:
# Main figure


### 6.5 Features for integration
Variables from this workstream that may help predict the target. Rules:
- Keys: `match, half, minute, side` (team level) or the same plus player ID (player level).
- Column names start with `B_`.
- Use only information available **up to and including** that minute.
- Write a one-line description of each feature.

In [ ]:
# Feature table and its registration


### 6.6 Takeaways
Two or three sentences, each with its evidence label and the number behind it. Add each one to the results log (Part 11).

## 7 · Workstream C - chances & possession changes
**Owner:** Pablo Saraiva

**Question:** What precedes shots and possession changes, and can they be predicted seconds ahead?

Fill in before running the main analysis:

| | |
|---|---|
| Hypothesis | |
| Unit of analysis | |
| Method | |
| Controls | |
| Rejection rule | What result would make us drop the hypothesis |
| Expected figure | |

Starting points: team control and ball location in the seconds before a shot or a possession change, compared with random moments; rates of possession changes by zone and match phase.

### 7.1 Exploration
First look at the variables: distributions, outliers, differences between matches and positions. Nothing here is a finding yet.

In [ ]:
# Exploration


### 7.2 Main analysis
The analysis that answers the question, using the method written above. Report effect sizes with confidence intervals, not only p-values.

In [ ]:
# Main analysis


### 7.3 Robustness
Does the result survive: each match separately, leave-one-match-out, alternative definitions from the decision log, a placebo or shuffled comparison, correction for multiple tests?

In [ ]:
# Robustness checks


### 7.4 Figure
One figure that a coach could read in ten seconds. Title states the finding; axes have units.

In [ ]:
# Main figure


### 7.5 Features for integration
Variables from this workstream that may help predict the target. Rules:
- Keys: `match, half, minute, side` (team level) or the same plus player ID (player level).
- Column names start with `C_`.
- Use only information available **up to and including** that minute.
- Write a one-line description of each feature.

In [ ]:
# Feature table and its registration


### 7.6 Takeaways
Two or three sentences, each with its evidence label and the number behind it. Add each one to the results log (Part 11).

## 8 · Integration
**Owner:** Leonardo · **Delivers:** one feature table at team × minute level with every registered feature, plus a registry of who built what.

- Player-level features are aggregated to the team (sum or mean, stated by the owner).
- Merges are on `match, half, minute, side`; duplicated keys or duplicated column names stop the merge.
- The registry lists: feature set, owner, number of features, description.

In [ ]:
# Feature registry


In [ ]:
# Merge into one feature table


## 9 · Prediction
**Owners:** Monica (models) · Pablo (validation) · **Delivers:** a model that beats a simple baseline on matches it never saw, frozen before the holdout match is opened.

### 9.1 Target
What we predict, at what unit and horizon, and the metric. This must match the course's holdout evaluation (decision D7, W6 task owned by Pablo). Until confirmed, use one regression and one classification example to build the pipeline.

In [ ]:
# Target definition(s)


### 9.2 Supervised table
For each team and half: the target is computed over the next *h* minutes; features come from the current and previous minutes only (lags, rolling means). Rows without a future value are dropped. State *h* and the lags used.

In [ ]:
# Supervised table: target ahead, features from the past


### 9.3 Validation protocol
- **Leave-one-match-out**: train on five matches, score on the sixth, repeat six times (D8).
- **Hyperparameters tuned inside the training matches only** (nested), never on the held-out match.
- Report the score of each held-out match, not only the average.

In [ ]:
# Cross-validation splits


### 9.4 Baselines
A model only counts if it beats these on held-out matches:
- Training mean (regression) or base rate (classification).
- Persistence: the current value carried forward.

In [ ]:
# Baselines


### 9.5 Models
From simple to complex: regularised linear / logistic regression, then gradient-boosted trees, then an ensemble of the two. Record each model's settings.

In [ ]:
# Models


### 9.6 Evaluation and uncertainty
- Regression: MAE, RMSE, R². Classification: log loss, Brier score, ROC AUC.
- Skill vs baseline with a bootstrap over held-out matches (the match is the unit that generalises).
- How many of the six matches the model beats the baseline in.

In [ ]:
# Scores per held-out match and bootstrap


### 9.7 Calibration and feature importance
- Calibration: predicted vs observed frequency, held-out predictions only.
- Importance: permutation importance computed on held-out matches, never on training data.

In [ ]:
# Calibration


In [ ]:
# Permutation importance


### 9.8 Model selection and freeze
Choose the model by held-out performance, write down why, then fit it on all six matches and save it with the feature list and configuration. After this cell, **no feature, parameter or model changes** (W10 milestone).

In [ ]:
# Final fit and freeze


## 10 · Holdout inference
**Owner:** Pablo · **Delivers:** predictions on the unseen match in the format the course asks for.

Steps:
1. Put the new match's three files in the data folder and open the holdout guard for that match only.
2. Run Parts 2–4 for it with the same code and parameters; all checks must pass.
3. Rebuild every workstream's features for it (same definitions as 5.5, 6.5, 7.5).
4. Load the frozen model, predict, and save the file to submit.

In [ ]:
# Build tables and features for the holdout match


In [ ]:
# Predict with the frozen model and save the submission


## 11 · Results log
**Owner:** all · One row per claim that may go into the presentation. Claims without a row are not presented.

| Column | Content |
|---|---|
| Workstream | A, B, C, Prediction |
| Claim | One sentence |
| Evidence label | One of the five labels |
| Estimate and 95% CI | With units |
| n | Players, minutes, events or matches |
| Method | Short description and the cell where it is computed |

In [ ]:
# Results log table


## 12 · Conclusions & limitations
**Owner:** all · Written last, after the team review.

- **Answer to the project question**, in three sentences, with evidence labels.
- **What the model does and does not do** on unseen matches.
- **Limitations:** six matches, five involving the same home team; observational data; simplified pitch-control model; possession from the DFL flag.
- **Next steps** we would take with more data.